# Observability and tracing with LangSmith

Instrument a real agent first, then inspect traces and attach evaluations. A trace is one end-to-end request; its child runs capture model, tool, retrieval, and graph steps.


## Trace a LangGraph agent

LangChain and LangGraph calls are traced automatically when `LANGSMITH_TRACING=true`. Use `@traceable` for application code outside the framework. Keep secrets and raw sensitive payloads out of trace inputs.


In [ ]:
# Import the dependencies used by this example.
import os
from langchain.agents import create_agent
from langchain.tools import tool
from langsmith import traceable

# Configure these in your shell, not in the notebook:
# LANGSMITH_TRACING=true
# LANGSMITH_API_KEY=...
# LANGSMITH_PROJECT=learn-agents

@tool
@traceable(name="inventory-lookup", run_type="tool")
# Define the data shape and small operations before running them.
def inventory_lookup(sku: str) -> dict:
    """Return stock for one SKU."""
    return {"sku": sku, "in_stock": 7}

# Configure the framework object; this line prepares it but may not execute it yet.
agent = create_agent(
    model="openai:gpt-5.6-sol",
    tools=[inventory_lookup],
    system_prompt="Answer inventory questions using the tool; never invent stock levels.",
)

# Execute the configured model or workflow with the input below.
result = agent.invoke({"messages": [{"role": "user", "content": "Is SKU-42 in stock?"}]})
result["messages"][-1].content


## Evaluate production behavior

Use offline datasets before release and online evaluators on sampled production traces afterward. Alert on a sustained score or latency shift, not a single noisy run; route failed traces back into a regression dataset.


In [ ]:
# Import the dependencies used by this example.
from langsmith import Client

client = Client()

# Define the data shape and small operations before running them.
def grounded(outputs: dict, reference_outputs: dict) -> dict:
    answer = outputs["answer"].lower()
    expected = reference_outputs["required_fact"].lower()
    return {"key": "grounded", "score": int(expected in answer)}

# Execute the configured model or workflow with the input below.
experiment = client.evaluate(
    lambda inputs: {"answer": agent.invoke({"messages": inputs["messages"]})["messages"][-1].content},
    data="inventory-regression",
    evaluators=[grounded],
    experiment_prefix="inventory-agent-v1",
)
experiment


## Production check

Confirm each trace includes the user-visible outcome, tool arguments after redaction, errors, latency, token usage, model identifier, and feedback. Use a stable project name and tags so releases can be compared.
